_Células em Markdown traduzidas por mim comentários originais realizados em comentários nas células de código._

# Reinforcement Learning
_(aprendizado por reforço)_

Vamos descrever o "problema do táxi". Nós queremos construir um taxi que dirige sozinho e que consegue buscar passageiros em uma das localizações marcadas, levá-los até uma outra localização e chegar lá o mais rápido possível enquanto evita obstáculos. 

O AI Gym permite criarmos o ambiente rapidamente:

In [1]:
# import gymnasium as gym
import gym
import random

random.seed(1234)

streets = gym.make("Taxi-v3", render_mode='ansi').env # Ajustado conforme descrito no card
streets.reset() # Ajustado conforme descrito no card
print("\n" + streets.render()) # Mostra o mapa


+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+




Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Vamos definir o que estamos vendo aqui:
- R, G, B, e Y são localizações de embarque ou desembarque.
- A letra em AZUL indica onde o táxi precisa buscar um passageiro.
- A letra em MAGENTA indica para onde o passageiro quer ir.
- As linhas retas `|` indicam paredes que o táxi não pode atravessar.
- O retângulo preenchido representa o próprio táxi, é amaralo quando está vazio e verde quando está carregando um passageiro.

Nosso pequeno mundo aqui, que chamamos de `streets` é um _grid_ 5x5. O estado do mundo pode ser definido a qualquer momento por:
- Onde o táxi está (uma das 5x5 = 25 localizações)
- Qual é o destino atual (4 possibilidade)
- Onde está o passageiro (5 possibilidades: em um dos destinos, ou dentro do táxi)

Então temos um total de 25 x 4 x 5 = 500 possibilidades de estado que podem descrever o nosso mundo.

Para cada estado há seis ações possíveis:
- Andar para Sul, Lest, Norte ou Oeste.
- Embarcar o passageiro.
- Desembarcar o passageiro.

Q-Learning será definido utilizando as seguintes recompensas ou penalidades para cada estado:
- Um desembarque de sucesso vale uma recompensa de +20 pontos.
- Cada passo realizado enquanto dirige com um passageiro resulta em penalidade de -1 ponto.
- Embarcar ou Desembarcar em um lugar ilegal resulta em uma penalidade de -10 pontos.

Mover-se através de uma parede não é permitido de nenhuma maneira.

Vamos definir o estado inicial, com a localização do táxi em (2, 3), o passageiro na localização 2, e o destino na localização 0:

In [2]:
initial_state = streets.encode(2, 3, 2, 0) # Defini o estado inicial 

streets.s = initial_state # atribui o estado ao ambiente `streets`

print('\n' + streets.render())


+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+




Vamos examinar a tabela de recompensas para o estado inicial:

In [3]:
streets.P[initial_state]

{0: [(1.0, 368, -1, False)],
 1: [(1.0, 168, -1, False)],
 2: [(1.0, 288, -1, False)],
 3: [(1.0, 248, -1, False)],
 4: [(1.0, 268, -10, False)],
 5: [(1.0, 268, -10, False)]}

Assim que vamos interpretar o estado - cada linha representa uma ação possível nesse estado: se mover para Sul, Norte, Leste, Oeste, embarcar ou desembarcar. Os quatro valores indicados em cada linha são a probabilidade definida para aquela ação, o próximo estado que resulta daquela ação, a recompensa daquela ação, e se a ação resulta em um desembarque com sucesso.

Se por exemplo, se mover norte deste estado irá colocar o táxi no estado número 368, resulta em uma penalidade de -1 por levar tempo, e não resulta em um desembarque de sucesso.

Então vamos fazer o Q-Learning! Primeiro precisamos treinar o nosso modelo. Basicamente, nós iremos treinar em mais de 10 mil corridas de táxi simuladas. Para cada corrida, vamos andar no tempo com uma chance de 10% a cada passo de escolher um passo aleatório, passo exploratório ao invés de utilizar o valor Q aprendido para guiar a decisão.

In [4]:
import numpy as np

q_table = np.zeros([streets.observation_space.n, streets.action_space.n]) # Tabela de valores Q matriz dos valores entre
                                                                            # a possibilidade de estados e ações que 
                                                                            # podem ser tomadas.

learning_rate = 0.1
discount_factor = 0.6 # Desconto utilizado ao definir uma ação boa ou ruim.
exploration = 0.1 # _epsilon term_
epochs = 10000 # Quantidade de corridas.

for taxi_run in range(epochs): # Simula 10 mil corridas.
    state = streets.reset()[0]
    done = False

    while not done:
        random_value = random.uniform(0, 1)
        if (random_value < exploration): # Como a decisão será tomada de acordo com o _epsilon term_ (MDP).
            action = streets.action_space.sample() # Escolhe uma ação aleatória.
        else:
            action = np.argmax(q_table[state]) # Usa a ação com o maior valor Q, as ações são definidas pelas colunas
                                                # de 0 a 5, assim retorna a ação com maior valor Q na linha.
            
        next_state, reward, done, other, info = streets.step(action) # Realiza a ação que foi decidida. Aqui temos as
                                                                # informações do próximo estado `(next_state), 
                                                                # a recompensa (`reward`), se a viagem foi concluída
                                                                # (`done`), e demais informações (`info`).
                                                                # 'other' foi adicionado pois existe outro valor ao
                                                                # deserializar a tupla que não consta na aula.

        prev_q = q_table[state, action] # Acessa a ação tomada anteriormente
        
        next_max_q = np.max(q_table[next_state]) # Recupera o valor máximo da próxima ação que pode ser tomada 
                                                    # para definir se a ação que foi escolhida foi ou não a melhor.
        
        new_q = (1 - learning_rate) * prev_q + learning_rate * (reward + discount_factor * next_max_q) # calcula o novo
                                                                                                        # valor de Q,
                                                                                                        # com base no 
                                                                                                        # que foi 
                                                                                                        # aprendido.
        
        q_table[state, action] = new_q # o Q foi calculado com base no `learning_rate` vezes o valor de Q anterior que
                                        # essa ação possuia somado com o `learning_rate` vezes: 
                                        # a soma da recompensa da ação (`reward`) e fator de desconto vezes
                                        # e o maior valor Q da próxima ação possível por conta da ação tomada.
        
        state = next_state

# OBS: Se eu executo o notebook reiniciando o kernel e executando todas as células o erro acontecia:
# module 'numpy' has no attribute 'bool8'
# para isso instalei a versão 1.26.4 do NumPy o que pode gerar conflito com as dependências do pandas e do SciPy mas 
# não afetou a execução do notebook.

/Users/ricardovessaro/venvs/relatorio-11/lib/python3.14/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


Agora temos uma tabela de valores Q que pode rapidamente ser utilizada para determinar o próximo passo mais 
otimizado dado um estado! Vamos conferir o estado incial acima:

In [5]:
q_table[initial_state]

array([-2.41491177, -2.4161591 , -2.41097747, -2.3639511 , -7.53635394,
       -7.27103726])

O menor valor de Q corresponde com a ação "ir para oeste", o que faz sentido - essa é a rota mais rápida para o
destino partindo daquele ponto. Parece que funcionou, vamos ver isso em ação!

In [6]:
from IPython.display import clear_output
from time import sleep 

for tripnum in range(1, 11):
    state = streets.reset()[0]

    done = False

    while not done:
        action = np.argmax(q_table[state]) # Agora com o treinamento podemos simplesmente escolher a ação que possui 
                                            # uma recompensa maior.
        next_state, reward, done, other, info = streets.step(action)
        clear_output(wait=True) # Para limpar a saida do console.
        print("Trip number " + str(tripnum))
        print('\n'+ streets.render())
        sleep(.5)
        state = next_state

    sleep(2)

Trip number 10

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (Dropoff)



# Seu desafio

Modifique o bloco acima para rastrear o total de passos e use isso como uma métrica para definir o quão bom o sistema Q-Learn é. Você pode querer aumentar o número de viagens simuladas, e remover a chamada `sleep()` para executar em mais amostras.

In [7]:
from IPython.display import clear_output
from time import sleep 

def render_showing_print_steps(q_table, render=True):
    if not render:
            print(f'render_showing_print_steps called')
        
    steps = 0
    for tripnum in range(1, 11):
        if not render:
            print(f'Running trip number {tripnum}...')
            
        state = streets.reset()[0]
    
        done = False
    
        while not done:
            action = np.argmax(q_table[state]) # Agora com o treinamento podemos simplesmente escolher a ação que possui 
                                                # uma recompensa maior.
            next_state, reward, done, other, info = streets.step(action)

            
            steps += 1 # Modificação para mostrar os passos
            
            if render:
                clear_output(wait=True) # Para limpar a saida do console.
                print("Trip number " + str(tripnum))
                print('\n'+ streets.render())
                print(f'Steps: {steps}') # Modificação para mostrar os passos
            #sleep(.5)
                sleep(.4)
            
            state = next_state

        if not render:
            print(f'Finished trip number {tripnum}, actual steps are {steps}')

        #sleep(2)

    if not render:
            print(f'render_showing_print_steps finished')

    return steps

render_showing_print_steps(q_table)

Trip number 10

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (Dropoff)

Steps: 130


130

Agora, tente experimentar com hiperparametros. O quanto o número de repetições (`epoch`) podem ser utilizados
antes do modelo começar a ter dificultadade? Você consegue utilizar melhores `learning_rates`, `discount_factors`, ou `exploration_factors` para fazer o treino ser mais eficiente? O _exploration_ vs. _exploitation_ _rates_ em particular são interessantes de experimentar.

In [14]:
# Duplicado o fonte para realizar as alterações sem mudar o fonte da aula

import numpy as np

def fit(learning_rate = 0.1, discount_factor = 0.6 , exploration = 0.1, epochs = 10000):
    print('Fitting Q-Learning...')

    qq_table = np.zeros([streets.observation_space.n, streets.action_space.n]) # Tabela de valores Q matriz dos valores entre
                                                                                # a possibilidade de estados e ações que 
                                                                                # podem ser tomadas.
    for taxi_run in range(epochs): # Simula 10 mil corridas.
        state = streets.reset()[0]
        done = False
    
        while not done:
            random_value = random.uniform(0, 1)
            if (random_value < exploration): # Como a decisão será tomada de acordo com o _epsilon term_ (MDP).
                action = streets.action_space.sample() # Escolhe uma ação aleatória.
            else:
                action = np.argmax(qq_table[state]) # Usa a ação com o maior valor Q, as ações são definidas pelas colunas
                                                    # de 0 a 5, assim retorna a ação com maior valor Q na linha.
                
            next_state, reward, done, other, info = streets.step(action) # Realiza a ação que foi decidida. Aqui temos as
                                                                    # informações do próximo estado `(next_state), 
                                                                    # a recompensa (`reward`), se a viagem foi concluída
                                                                    # (`done`), e demais informações (`info`).
                                                                    # 'other' foi adicionado pois existe outro valor ao
                                                                    # deserializar a tupla que não consta na aula.
    
            prev_q = qq_table[state, action] # Acessa a ação tomada anteriormente
            
            next_max_q = np.max(qq_table[next_state]) # Recupera o valor máximo da próxima ação que pode ser tomada 
                                                        # para definir se a ação que foi escolhida foi ou não a melhor.
            
            new_q = (1 - learning_rate) * prev_q + learning_rate * (reward + discount_factor * next_max_q) # calcula o novo
                                                                                                            # valor de Q,
                                                                                                            # com base no 
                                                                                                            # que foi 
                                                                                                            # aprendido.
            
            qq_table[state, action] = new_q # o Q foi calculado com base no `learning_rate` vezes o valor de Q anterior que
                                            # essa ação possuia somado com o `learning_rate` vezes: 
                                            # a soma da recompensa da ação (`reward`) e fator de desconto vezes
                                            # e o maior valor Q da próxima ação possível por conta da ação tomada.
            
            state = next_state

    print('Fitting Q-Learning finished')
    return qq_table

# OBS: Se eu executo o notebook reiniciando o kernel e executando todas as células o erro acontecia:
# module 'numpy' has no attribute 'bool8'
# para isso instalei a versão 1.26.4 do NumPy o que pode gerar conflito com as dependências do pandas e do SciPy mas 
# não afetou a execução do notebook.

In [15]:
new_q_table = fit(learning_rate = 0.2, discount_factor = 0.6 , exploration = 0.1, epochs = 10000)

steps = render_showing_print_steps(new_q_table, render=False)
print(f'Steps {steps}')

Fitting Q-Learning...
Fitting Q-Learning finished
render_showing_print_steps called
Running trip number 1...
Finished trip number 1, actual steps are 8
Running trip number 2...
Finished trip number 2, actual steps are 18
Running trip number 3...
Finished trip number 3, actual steps are 35
Running trip number 4...
Finished trip number 4, actual steps are 47
Running trip number 5...
Finished trip number 5, actual steps are 62
Running trip number 6...
Finished trip number 6, actual steps are 73
Running trip number 7...
Finished trip number 7, actual steps are 83
Running trip number 8...
Finished trip number 8, actual steps are 96
Running trip number 9...
Finished trip number 9, actual steps are 109
Running trip number 10...
Finished trip number 10, actual steps are 125
render_showing_print_steps finished
Steps 125


In [16]:
new_q_table = fit(learning_rate = 0.2, discount_factor = 0.6 , exploration = 0.3, epochs = 10000)

steps = render_showing_print_steps(new_q_table, render=False)
print(f'Steps {steps}')

Fitting Q-Learning...
Fitting Q-Learning finished
render_showing_print_steps called
Running trip number 1...
Finished trip number 1, actual steps are 14
Running trip number 2...
Finished trip number 2, actual steps are 27
Running trip number 3...
Finished trip number 3, actual steps are 43
Running trip number 4...
Finished trip number 4, actual steps are 55
Running trip number 5...
Finished trip number 5, actual steps are 71
Running trip number 6...
Finished trip number 6, actual steps are 84
Running trip number 7...
Finished trip number 7, actual steps are 98
Running trip number 8...
Finished trip number 8, actual steps are 110
Running trip number 9...
Finished trip number 9, actual steps are 125
Running trip number 10...
Finished trip number 10, actual steps are 139
render_showing_print_steps finished
Steps 139


In [17]:
new_q_table = fit(learning_rate = 0.3, discount_factor = 0.6 , exploration = 0.3, epochs = 10000)

steps = render_showing_print_steps(new_q_table, render=False)
print(f'Steps {steps}')

Fitting Q-Learning...
Fitting Q-Learning finished
render_showing_print_steps called
Running trip number 1...
Finished trip number 1, actual steps are 12
Running trip number 2...
Finished trip number 2, actual steps are 23
Running trip number 3...
Finished trip number 3, actual steps are 34
Running trip number 4...
Finished trip number 4, actual steps are 42
Running trip number 5...
Finished trip number 5, actual steps are 56
Running trip number 6...
Finished trip number 6, actual steps are 68
Running trip number 7...
Finished trip number 7, actual steps are 84
Running trip number 8...
Finished trip number 8, actual steps are 93
Running trip number 9...
Finished trip number 9, actual steps are 110
Running trip number 10...
Finished trip number 10, actual steps are 119
render_showing_print_steps finished
Steps 119


In [18]:
# new_q_table = fit(learning_rate = 0.1, discount_factor = 0.6 , exploration = 0.3, epochs = 1000)

# steps = render_showing_print_steps(new_q_table, render=False)
# print(f'Steps {steps}')
# Comentado pois com 1 mil simplesmente se perde

In [19]:
# O melhor resultado explorado foi learning_rate = 0.3, discount_factor = 0.6 , exploration = 0.3, epochs = 10000 que 
# finalizou em 119 passos em comparação os demais. Os resultados podem variar. Observando os valores o learning rate maior
# faz com que o modelo aprenda mais rápido e o exploration façã com que a chance de escolher um valor aleatório é maior.